# Introduction to Fourier Methods

James Johnson, CIERA/Northwestern

LSST Discovery Alliance Data Science Fellowship Program

Session 27: Time-Series Analysis

November 16-20, 2026

Caltech/IPAC, Pasadena, CA

# Fourier Methods

* Form the backbone of much that will come later this week
    -  A lot easier to understand, e.g., Lomb-Scargle periodograms if you understand Fourier methods
* Still very relevant for multiple purposes
    - Deconvolving point spread functions from images
    - Computing gradients of graviational potentials in simulations

These lectures are going to hit concepts *hard* in an effort to make the next things easy to understand.

# Review: Fourier's Theorem

**Any periodic signal can be represented as a series of sines and cosines.**

Any function $f(x)$ with a period $T$ can be written as:

$$
\begin{align}
f(x) &= A_0 + \sum_{n=1}^\infty \left[
A_n \cos \left( \frac{2n\pi x}{T} \right) + B_n \sin \left(\frac{2n\pi x}{T} \right)
\right]
\\
A_0 &= \frac{1}{T} \int_0^T f(x) dx
\\
A_n &= \frac{2}{T} \int_0^T f(x) \cos \left(\frac{2n\pi x}{T}\right) dx
\\
B_n &= \frac{2}{T} \int_0^T f(x) \sin \left(\frac{2n\pi x}{T}\right) dx
\end{align}
$$

## A Quick Example: A Square Wave

The square wave looks like a top-hat function, but repeatedly goes back and forth between two values. One definition:

$$
f(x) = \begin{cases}
1 & (x \% 2 \leq 1)
\\
-1 & (x \% 2 > 1)
\end{cases}
$$

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings("ignore") # they just take up space on slides

In [ ]:
def square_wave(x):
    r"""
    The square wave, which is essentially a repeating top-hat function.

    Parameters
    ----------
    x : `float`
        A real number.

    Returns
    -------
    1 if the integer floor of x is an even number. -1 otherwise.
    """
    if x % 2 <= 1: # floor(x) is even
        return 1
    else: # floor(x) is odd
        return -1

In [ ]:
xvals = np.linspace(0, 6, 1000)
yvals = [square_wave(x) for x in xvals]
plt.plot(xvals, yvals)
plt.xlabel("x")
plt.ylabel("Square Wave")

For a simple waveform like the square wave, the coefficients $A_n$ and $B_n$ of the Fourier series can be computed analytically:

$$
\begin{align}
A_0 &= 0
\\
A_n &= 0
\\
B_n &= \frac{4}{\pi \left(2n - 1\right)}
\end{align}
$$

which results in the following Fourier series:

$$
f(x) = \frac{4}{\pi} \sum_{n = 1}^\infty \frac{\sin \left(\left(2n - 1\right) \pi x\right)}{2n - 1}.
$$

It is not a coincidence that $A_n = 0$ for all $n$ with how we have defined the square wave. Any ideas why?

As we have defined it above, the square wave is an "odd" function, meaning that $f(-x) = -f(x)$. Sine is an odd function, while cosine is an even function (i.e., $f(-x) = f(x)$). All odd functions have a Fourier series with $A_n = 0$, and all even functions have a Fourier series with $B_n = 0$ because of these properties of the sine and cosine functions.

In [ ]:
def square_wave_fourier_series(x, n = 10):
    r"""
    A Fourier series representing the square wave.

    Parameters
    ----------
    x : `float`
        A real number.
    n : `int` [default : 10]
        The integer number of sinusoids to add up to represent the
        square wave.

    Returns
    -------
    The n'th order Fourier series approximation of the square wave
    at the value of x.
    """
    if n:
        # recursive solution -- add the term for n, then call itself
        # for n - 1, all the way down to n = 0
        return 4 / np.pi * np.sin(
            (2 * n - 1) * np.pi * x) / (
            2 * n - 1) + square_wave_fourier_series(x, n = n - 1)
    else: # base case for the recursive solution
        return 0

In [ ]:
xvals = np.linspace(0, 6, 1000)
for n in [2, 5, 10, 20]:
    yvals = [square_wave_fourier_series(x, n = n) for x in xvals]
    plt.plot(xvals, yvals, label = "N = %d" % (n))
plt.legend(loc = "center right")
plt.xlabel("x")
plt.ylabel("Square Wave Fourier Series")
plt.show()

In practice, Fourier series do not come up frequently in astronomy because they require you to already know the functional form of the wave. For many purposes, constraining the waveform itself is the primary goal. Typically, we are asking questions closer to the following:

- "For some number of measurements of a signal, what is the underlying waveform?"
- "What are the characteristic frequencies driving most of the oscillation?"
- "What are the amplitudes associated with those frequencies?"

The Fourier transform is the classical method used to answer these sorts of questions.

# The Fourier Transform: Core Concepts

Any time-varying signal can be described in two ways:
- As a function of time, $f(t)$, or "in the time domain"
- As a function of frequency, $\tilde f(k)$, or "in the frequency domain"

It is generally useful to think of $f(t)$ and $\tilde f(k)$ as being two different representations of the same function. The *Fourier transform* equations go back and forth between the two:

$$
\begin{align}
\tilde f(k) &= \int_{-\infty}^\infty f(t) e^{-2\pi i kt} dt
\\
f(t) &= \int_{-\infty}^\infty \tilde f(k) e^{2\pi i kt} dk
\end{align}
$$

## Regarding Notation

Physicists and mathematicians love to use different notation than one another. As astronomers, many of us are probably more familiar with writing periodic functions in terms of an *angular frequency*, $\omega = 2\pi k$. Under this notation, the Fourier transform becomes

$$
\begin{align}
\tilde f(\omega) &= \int_{-\infty}^\infty f(t) e^{-i\omega t} dt
\\
f(t) &= \frac{1}{2\pi} \int_{-\infty}^\infty \tilde f(\omega) e^{i\omega t} d\omega
\end{align}
$$

If you have read the popular textbook *Numerical Recipes* by Press, Teukolsky, Vetterling \& Flannery, you might also notice that they invert the sign convention for the complex exponentials in the integrands:

$$
\begin{align}
\tilde h(k) &= \int_{-\infty}^\infty h(t) e^{2\pi ikt} dt
\\
h(t) &= \int_{-\infty}^\infty \tilde h(k) e^{-2\pi ikt} dt.
\end{align}
$$

Be aware that you might see slightly different definitions of the Fourier transform! The key detail is that the *interpretation* is the same between definitions. Your choice of any one convention is inconsequential, but your consistency in sticking to that convention for a given set of calculations *is* consequential.

## Fourier Transforms in Action

The Simplest Periodic Signal: A Single Sinusoid
--

The following sinusoid has a period of 1 (let's say that's in days):

$$
f(t) = \sin(2 \pi t).
$$

In [ ]:
simple_sinusoid = lambda t: np.sin(2 * np.pi * t)
times = np.linspace(0, 3, 1500)
signal = [simple_sinusoid(t) for t in times]
plt.plot(times, signal)
plt.xlabel(r"$t$ [days]")
plt.ylabel(r"$\sin(2\pi t)$")
plt.show()

Let's take the Fourier transform of this sinusoid. In order to better keep track of our variables, I'll generalize the definition of this sinusoid to

$$
f(t) = A \sin (2\pi k_0 t),
$$

where $A$ and $k_0$ are the amplitude and frequency of the wave, respectively. The specific case we've plotted uses $A = 1$ and $k_0 = 1$ day$^{-1}$. It's Fourier transform:

$$
\tilde f(k) = \int_{-\infty}^\infty A \sin (2\pi k_0 t) e^{-2\pi ikt} dt.
$$

At this point, it is helpful to write the sine function in terms of complex exponentials:

$$
\sin (2\pi k_0 t) = \frac{e^{2\pi i k_0 t} - e^{-2\pi i k_0 t}}{2i},
$$

which allows the integral to be expressed purely in terms of exponential functions:

$$
\tilde f(k) = \frac{-Ai}{2} \lim_{q \rightarrow \infty} \left[
\int_{-q}^q e^{-2\pi i (k - k_0) t} dt - \int_{-q}^q e^{-2\pi i (k + k_0) t} dt
\right].
$$

Solving the integral and converting from complex exponentials back to sinusoids:

$$\tilde f(k) = -Ai \lim_{q \rightarrow \infty} \left[
\frac{\sin(2\pi (k - k_0) q)}{2\pi (k - k_0)} -
\frac{\sin(2\pi (k + k_0) q)}{2\pi (k + k_0)}
\right]
$$

## The Dirac Delta Function

This expression for $\tilde f(k)$ contains the limit definition of the Dirac delta function, $\delta (x)$. You're probably more familiar with this definition:

$$
\delta (x) = \begin{cases}
\infty & (x = 0)
\\
0 & (x \neq 0).
\end{cases}
$$

However, $\delta(x)$ can also be rewritten as:

$$
\delta (x) = \lim_{q \rightarrow \infty} \frac{\sin (qx)}{\pi x}.
$$

This definition comes from the *Dirichlet kernel* $D_n$, which refers to a set of periodic functions defined by

$$
D_n = 1 + \sum_{k = 1}^n \cos(kx) = \frac{\sin((n + \frac{1}{2})x)}{\sin(x/2)}.
$$

To visualize this relationship:

In [ ]:
def dirichlet_dirac_delta(x, q):
    r"""
    The limit definition of the dirac delta function.

    Parameters
    ----------
    x : `float`
        The x-coordinate.
    q : `int`
        The q'th order of the Dirichlet kernel to evaluate.

    Returns
    -------
    The limit definition of the dirac delta function, up to the
    q'th term.
    """
    if q:
        return np.cos(q * x) + dirichlet_dirac_delta(x, q - 1)
    else:
        return 1

In [ ]:
xvals = np.linspace(-2, 2, 1000)
for q in [5, 10, 20, 50]:
    yvals = [dirichlet_dirac_delta(x, q) for x in xvals]
    plt.plot(xvals, yvals, label = "q = %d" % (q))
plt.xlabel(r"$x$")
plt.ylabel(r"$D_q(x)$")
plt.legend(loc = "upper right")

This plot makes it clear that as $q \rightarrow \infty$, the function converges at $f(x) = 0$ for all values of $x \neq 0$ but diverges to $f(x) \rightarrow \infty$ at $x = 0$, which is the defining feature of the Dirac delta function.

Inserting this definition of $\delta (x)$ into our most recent expression for $\tilde f(k)$ simplifies the result:

$$
\tilde f(k) = -A\pi i \left[\delta(2\pi (k - k_0)) - \delta(2\pi (k + k_0))\right].
$$

The factor of $2\pi$ in both delta functions applies an inconsequential stretch to the $x$-coordinate in $\delta(x)$, so we can simply drop it:

$$
\tilde f(k) = -A\pi i \left[ \delta (k - k_0) - \delta (k + k_0) \right]
$$

## A Conceptually Important Fourier Transform Pair

Using the notation $f(t) \Leftrightarrow \tilde f(k)$ to denote pairs of functions and their Fourier transforms, the following is a pair that is particularly important conceptually:

$$
A\sin(2\pi k_0 t) \Leftrightarrow -A \pi i \left[ \delta(k - k_0) - \delta(k + k_0) \right].
$$

The Fourier transform of a single sinusoid is two delta functions: one at $k = k_0$, the frequency of the original sinusoid, and one at $k = -k_0$, its exact negative.

## Potentially Useful Properties

Certain identities and theorems follow from the definition of the Fourier transform:

| If... | then... |
| --- | --- |
| $f(t)$ is real | $\tilde f(-k) = \tilde f(k)^*$ | 
| $f(t)$ is imaginary | $\tilde f(-k) = -\tilde f(k)^*$ |
| $f(t)$ is even (i.e., $f(-t) = f(t)$) | $\tilde f(k)$ is also even |
| $f(t)$ is odd (i.e., $f(-t) = -f(t)$) | $\tilde f(k)$ is also odd |
| $f(t)$ is real and even | $\tilde f(k)$ is also real and even |
| $f(t)$ is imaginary and even | $\tilde f(k)$ is also imaginary and even |
| $f(t)$ is real and odd | $\tilde f(k)$ is imaginary and odd |
| $f(t)$ is imaginary and odd | $\tilde f(k)$ is real and odd |

## Potentially Useful Fourier Transform Pairs

$$
\begin{align}
f(at) &\Leftrightarrow \frac{1}{\left|a\right|} \tilde f\left(\frac{k}{a}\right) \qquad \text{(the "stretch theorem")}
\\
\frac{1}{\left|a\right|} f\left(\frac{t}{a}\right) &\Leftrightarrow \tilde f(ak)
\\
f(t - t_0) &\Leftrightarrow \tilde f(k) e^{-2\pi ikt_0} \qquad \text{(the "shift theorem")}
\\
f(t) e^{2\pi ik_0 t} &\Leftrightarrow \tilde f(k - k_0)
\end{align}
$$

## Convolution

Fourier transforms are extremely useful for convolving or deconvolving functions from one another. We'll see more of this as the week goes on.

The *convolution* of two functions, $f$ and $g$, is defined as:

$$
f * g = \int_{-\infty}^\infty f(\tau) g(t - \tau) d\tau.
$$

Deconvolving functions is much easier in the frequency domain than the time domain because of the following Fourier transform pair:

$$
f * g \Leftrightarrow \tilde f(k) \tilde g(k).
$$

This pair is so useful that it even has a name: the *convolution theorem*.

Common example: $f(t)$ is a signal you want to measure. The instrument you use to make the measurement imposes a response function, $g(t)$, such that what you measure is actually $(f * g)(t)$.

You can get $f(t)$ on its own, as desired, by exploiting the convolution theorem:

$$
f(t) = \int_{-\infty}^\infty \frac{\widetilde{(f*g)}(k)}{\tilde g(k)} e^{2\pi i k t} dk.
$$

Fourier methods are the backbone of most deconvolution algorithms for this reason (e.g., subtracting out camera point-spread-functions). More considerations go into the methodology when one starts worrying about the noise in the data.

## Fourier Transforms Numerically

Let's add a few sinusoids together to see a more complex example -- one that we can't do by hand (or at least wouldn't *want* to). We'll use a three-component sinusoid:

- The original simple one with a period of 1 day and amplitude of 1
- Another with half of the amplitude at a frequency of 3.6 per day
- Another with 1/5th the amplitude at a frequency of 14.9 per day

This three-component sinusoid will show up a lot throughout this lecture on Fourier series.

We'll use it, learn from it, add to it, pull it, twist it, bop it....

In [ ]:
def not_so_simple_sinusoid(t):
    r"""
    A three-component sinusoid that this notebook uses as a baseline
    example periodic function.

    Parameters
    ----------
    t : `float`
        Time, in arbitrary units (discussion assumes days).

    Returns
    -------
    The value of an arbitrary "signal" observed at the specified time.
    """
    return np.sin(2 * np.pi * t) + 0.5 * np.sin(
        7.2 * np.pi * t) + 0.2 * np.sin(29.8 * np.pi * t)

In [ ]:
sampling_interval = 3 / 1500
times = np.linspace(0, 3, 1500)
signal = [not_so_simple_sinusoid(t) for t in times]
frequencies = [(-len(signal) / 2 + _) / (
    len(signal) * sampling_interval) for _ in range(len(signal))]
plt.plot(times, signal)
plt.xlabel(r"Time [days]")
plt.ylabel("Signal")
plt.show()

### Discrete Fourier Transform

**DFT**: The most basic way to compute Fourier transform of discretely sampled data

Many of you have probably heard of or are familiar with the "Fast Fourier Transform" (FFT), which is an optimized version of the DFT.

| | |
| --- | --- |
| DFT | $\mathcal{O}(N^2)$ |
| FFT | $\mathcal{O}(N \log_2 N)$ |

At $N = 10^8$, the ratio between $\mathcal{O}(N^2)$ and $\mathcal{O}(N \log_2 N)$ is similar to the ratio between one second and one month.

... suffice it to say you should use the FFT for all *real* applications. Some simple by-hand DFTs are nonetheless instructive andstraight-forward.

Suppose we have a set of $N$ measurements of $f(t)$ at evenly spaced times. We can represent these data as:

$$
\begin{align}
\lbrace f(t_0), f(t_1), f(t_2), ..., f(t_{N - 1}) \rbrace &= \lbrace f(q \Delta t) \rbrace
\\
\lbrace t_1, t_2, t_3, ..., t_N \rbrace &= \lbrace q \Delta t \rbrace
\\
q &= \lbrace 0, 1, 2, ..., N - 1 \rbrace,
\end{align}
$$

where $\Delta t$ is the sampling interval. The sampling *frequency* or sampling *rate* would be $\Delta t^{-1}$.

In this "not-so-simple-sinusoid" example, we have $N = 1500$ samples of our odd-looking waveform over 3 days. The sampling interval is therefore $\Delta t = 3~\text{days} / 1500~\text{measurements} = 0.002~\text{days}$, or a little less than 3 minutes.

With $N$ measurements, we can only compute $\tilde{f}(k)$ at $N$ frequencies. Remember that we need to consider negative frequencies as well, so we need to compute both $\tilde f(k)$ and $\tilde f(-k)$. A Fourier transform of discretely sampled data is therefore restricted to the following frequency range:

$$
\begin{align}
k_n &= \frac{n}{N \Delta t}
\\
n &= \frac{-N}{2}, \frac{-N}{2} + 1, \frac{-N}{2} + 2, ..., \frac{N}{2}.
\end{align}
$$

Technically there are $N + 1$ values of $n$ here, but it turns out that $\tilde f(k)$ is identical at $n = -N/2$ and $n = +N/2$, so there are still $N$ unique values with information.

With discretely sampled data, we approximate the integral form of $\tilde f(t)$ as a summation:

$$
\begin{align}
\tilde f(k_n) &= \int_{-\infty}^\infty f(t) e^{-2\pi ik_n t} dt
\\
&\approx \sum_{q = 0}^{N - 1} f(t_q) e^{-2\pi i k_n t_q} \Delta t.
\end{align}
$$

In practice, modern FFT algorithms (e.g., NumPy, SciPy) compute $\tilde f(k)$ in a way that is dimensionless. 

With $k_n = n / N \Delta t$ and $t_q = q\Delta t$, one can substitute in $k_n t_q = q n / N$ and simply drop $\Delta t$ as a uniform scaling factor. We then arrive at the dimensionless DFT, which we'll denote as $\tilde f_n$:

$$
\tilde{f}_n = \sum_{q = 0}^{N - 1} f_q e^{-2\pi i q n / N},
$$

where $f_q = f(q\Delta t)$ is simply the $q$'th measurement.

A straightforward numerical implementation of this discrete Fourier transform solution:

In [ ]:
def dimensionless_dft(arr):
    r"""
    Compute the dimensionless discrete Fourier transform of the
    array `arr`.

    Parameters
    ----------
    arr : array-like (1-D)
        A sequence of floats to compute the Fourier transform of.

    Returns
    -------
    An array of equal length as `arr` representing its discrete
    Fourier transform.
    """
    ddft = []
    for idx in range(len(arr)): # one loop over frequencies ...
        n = -len(arr) / 2 + idx
        ddft.append(0)
        for q in range(len(arr)): # ... and one over times
            ddft[-1] += arr[q] * np.exp(-2j * np.pi * q * n / len(arr))
    return ddft

The $\mathcal{O}(N^2)$ scaling of the DFT is glaringly obvious in this implementation: there are two nested for-loops over the data.

In [ ]:
dft = dimensionless_dft(signal)
plt.plot(frequencies, dft)
plt.xlim([-20, 20])
plt.xlabel(r"$k$ [day^${-1}$]")
plt.ylabel(r"$\tilde f(k)$")
plt.show()

This Fourier transform looks odd because the solutions are complex, and we're only plotting the real component. A better way to visualize a Fourier transform is by evaluating the magnitude (a.k.a. modulus or absolute value) of each complex solution:

$$
c = a + bi \implies \left|c\right| = \sqrt{(a + bi)(a - bi)} = \sqrt{a^2 + b^2}.
$$

In python, this can be achieved by simply calling `abs` or `numpy.abs`.

Phase information gets encoded by "trading off" between $a$ and $b$ within $c = a + bi$. This can be understood by looking closely at the shift theorem.

The shift theorem introduces a factor of $e^{2\pi i}$ times the phase offset, hence its relation to phase information.

In [ ]:
absdft = [abs(_) for _ in dft]
plt.plot(frequencies, absdft)
plt.xlim([-20, 20])
plt.xlabel(r"$k$ [day$^{-1}$]")
plt.ylabel(r"$\left|\tilde{f}(k)\right|$")
plt.show()

Now the outcome is much clearer. We see the input wave with its component frequencies at $k = 1$ day$^{-1}$, $k = 3.6$ day$^{-1}$, and $k = 14.9$ day$^{-1}$, each showing up with a negative-frequency counterpart. We also see that the heights of these features scale with the relative amplitudes of the component waves that we put in.

## The Power Spectral Density

Negative frequencies are not physical. They arise as a mathematical solution to the Fourier transform, but for real data, they ultimately carry the same information as their positive frequency counterparts. The *one-sided power spectral density* (PSD) combines the positive and negative frequencies:

$$
P(k) = \left|\tilde f(k)\right|^2 + \left|\tilde f(-k)\right|^2 \qquad (0 \leq k \leq \infty).
$$

The PSD answers the question of "how much power" is contained in a certain frequency interval $k \rightarrow k + dk$.

The PSD for our toy sinusoid:

In [ ]:
psd = []
positive_frequencies = []
midpoint = len(absdft) // 2 # the index where I put a frequency of zero
for i in range(midpoint):
    positive_frequencies.append(frequencies[midpoint + i])
    psd.append(absdft[midpoint + i]**2 + absdft[midpoint - i]**2)
plt.figure(figsize = (3, 3))
plt.plot(positive_frequencies, psd)
plt.xlim([0, 20])
plt.xlabel(r"$k$ [day^${-1}$]")
plt.ylabel(r"$P(k)$")
plt.show()

## Filtering

A common task in time-series analysis, for many purposes, is to "filter out" or "remove" either high- or low-frequency features in the data. These features could be due to the instrument design, for example, and could be masking a signal of interest.

To filter time-series data, we'll need both Fourier transforms and inverse Fourier transforms. In similar fashion as the forward transformation, the discrete inverse Fourier transform follows from a numerical approximation of its exact integral form:

$$
\begin{align}
f(t) &= \int_{-\infty}^\infty \tilde f(k) e^{2\pi i k t} dt
\\
\implies f_q &\approx \frac{1}{N} \sum_{n = 0}^{N - 1} \tilde f_n e^{2\pi i q n / N}.
\end{align}
$$

*Beware* that there are different conventions when computing Fourier transforms and their inverses about where the factor of $N$ goes. The routines in `scipy.fft` are defined according to the following:

- `norm = "backward"` (default) The *inverse* Fourier transform is divided by $N$.
- `norm = "forward"` The Fourier transform itself is divided by $N$.
- `norm = "ortho"` Both the Fourier transform and its inverse are divided by $\sqrt{N}$.

Each of these implementations are perfectly valid and self-consistent, and the interpretations are the same between each of them. What's important is not that you choose one specific implementation, but that you remain *consistent* and carry one choice through all of your calculations.

An implementation of the inverse discrete Fourier transform:

In [ ]:
def dimensionless_ift(arr):
    r"""
    Compute the inverse discrete Fourier transform of the array `arr`.

    Parameters
    ----------
    arr : array-like (1-D)
        The input sequence of floats.

    Returns
    -------
    An array of equal length as `arr` representing its inverse
    discrete Fourier transform.
    """
    ift = []
    for q in range(len(arr)):
        ift.append(0)
        for idx in range(len(arr)):
            n = -len(arr) / 2 + idx
            ift[-1] += arr[idx] * np.exp(2j * np.pi * q * n / len(arr))
        ift[-1] /= len(arr)
    return ift

As expected, the inverse Fourier transform now returns the original waveform:

In [ ]:
ift = [_.real for _ in dimensionless_ift(dft)]
plt.plot(times, ift)
plt.xlabel(r"Time [days]")
plt.ylabel(r"f(t)")
plt.show()

Now lets filter certain frequency modes out of our signal. To do this, we'll simply set the frequencies above/below some threshold equal to zero in the frequency domain, then take the inverse Fourier transform back to the time domain.

In [ ]:
def simple_lowpass_filter(fk, freq, threshold):
    r"""
    A simple low-pass filter: tosses out high frequencies.

    Parameters
    ----------
    fk : array-like
        A discrete Fourier transform.
    freq : array-like
        The frequencies on which `fk` is sampled, matched
        component-wise.
    threshold : `float`
        The frequency above which oscillations will be
        removed.

    Returns
    -------
    An array that would be a copy of `fk` but has
    certain frequencies set to zero.
    """
    filtered = len(fk) * [0.]
    for i in range(len(freq)):
        filtered[i] = int(abs(freq[i]) <= threshold) * fk[i]
    return filtered

This simple step-function low-pass filter reveals the different oscillation modes of the not-so-simple-sinusoid:

In [ ]:
truncated1 = simple_lowpass_filter(dft, frequencies, 10)
truncated2 = simple_lowpass_filter(dft, frequencies, 2)

In [ ]:
fig, [ax1, ax2] = plt.subplots(figsize = (6, 3), nrows = 1, ncols = 2)
ax1.set_xlabel(r"$k$ [day$^{-1}$]")
ax1.set_ylabel(r"$\left|\tilde f(k)\right|$")
ax1.set_xlim([-20, 20])
ax2.set_xlabel(r"Time [days]")
ax2.set_ylabel(r"Signal")
ax1.plot(frequencies, absdft, c = 'k')
ax2.plot(times, signal, c = 'k')
ax1.plot(frequencies, [abs(_) for _ in truncated1], c = 'r')
ax1.plot(frequencies, [abs(_) for _ in truncated2], c = 'b')
ax2.plot(times, [_.real for _ in dimensionless_ift(truncated1)], c = 'r')
ax2.plot(times, [_.real for _ in dimensionless_ift(truncated2)], c = 'b')
plt.tight_layout()
plt.show()

Fitlering has many applications in time-series analysis. For example, one of the exercises uses filtering in the frequency domain to remove noise from a "noisified" version of the not-so-simple-sinusoid.

### Filtering and Convolution

Filtering is an application of the convolution theorem! The relationship can be seen by looking closely at the calculations we just did:

$$
\tilde{f}_\text{filtered}(k) = \tilde{h}(k) \tilde{f}_\text{input}(k),
$$

where $\tilde{f}_\text{input}$ is the Fourier transform of the input signal, and $\tilde{h}(k)$ is the filter in the frequency domain (in this case, simple step-function or "threshold" filters we've implemented here).

To filter a signal, we're ultimately colvolving the signal with some function using transforms -- we just defined the filter in the frequency domain instead of the time domain.

## Sampling

Let's say that our original signal has some very high frequency component built in.

Our examples so far drew 1500 measurements of the signal over 3 days, which corresponds to a measurement every ~3 minutes. Let's suppose that this insane variable star has a pulsation mode with a period of only one minute!

Perhaps our signal is an extremely odd-looking light curve from some new, exotic type of variable star, and isolating this high-frequency mode would confirm your theory and totally dunk on your competitors.

Here's the problem though -- this oscillation mode is at such high frequency that it's faster than our measurements!

An extension of our not-so-simple-sinusoid with an additional oscillation mode at a frequency of 1 per minute:

In [ ]:
def super_high_frequency(t):
    r"""
    The "signal" that we would measure if there is an extremely high
    frequency mode of oscillation on top of our original
    three-component sinusoid.

    Parameters
    ----------
    t : `float`
        Time :math:`t` in arbitrary units (discussion assumes days).

    Returns
    -------
    The value of this slightly different but still very arbitrary
    "signal" at the specified time.
    """
    return not_so_simple_sinusoid(t) + 0.15 * np.sin(2880 * np.pi * t)

In [ ]:
new_signal = [super_high_frequency(t) for t in times]
plt.plot(times, new_signal)
plt.xlabel(r"Time [days]")
plt.ylabel(r"Signal $f(t)$")
plt.show()

We're not measuring the flux from this variable star frequently enough to capture one cycle of oscillations...

... but maybe observing it for long enough will give us enough *information* to isolate the period?

... I can let the magic of Fourier transforms carry me all the way to my Nobel Prize!

... right!?

The Fourier transform of our new signal:

In [ ]:
new_dft = dimensionless_dft(new_signal)
plt.figure(figsize = (3, 3))
plt.plot(frequencies, [abs(_) for _ in new_dft])
plt.xlim([-80, 80])
plt.xlabel(r"$k$ [day$^{-1}$]")
plt.ylabel(r"$\left|\tilde f(k)\right|$")
plt.show()

Aha! There it is! The high frequency peak in my frequency spectrum! Right there at $k \approx 60$ day$^{-1}$!

... wait a minute... only 60 cycles per day?

My star is supposed to have a cycle every minute.... That's $1,400$ per day....

Where is the magic of Fourier transforms that I've known and loved!? Why do you fail me now!?

### Aliasing and the Sampling Theorem

The sampling theorem states that one must make a measurement at least twice per oscillation cycle in order to accurately characterize a given oscillation mode. This threshold corresponds to the *Nyquist critical frequency*:

$$
k_c = \frac{1}{2 \Delta t},
$$

where $\Delta t$ is our sampling interval.

In other words, the cost of accessing a given frequency range is capturing at least one peak and at least one trough of the sinusoids with that frequency.

#### The Good News


If your signal has oscillation modes that are all at frequencies below the Nyquist limit, then they are sampled sufficiently well that *your signal is fully characterized*.

#### The Bad News

If your signal has oscillation modes at frequencies *above* the Nyquist limit, then the Fourier transform will spurriously move them all into the $-k_c \leq k \leq +k_c$ range. This phenomenon is called *aliasing*. There is little you can do to remove aliased power once you have discretely sampled a signal.

Another way to think about this: any intrinsic oscillatory behavior in your data *will* influence its Fourier transform, and if it can't do so at the correct frequency, it will do so at an incorrect one.

**Aliasing in Action**: If we plot the inverse Fourier transform of our frequency spectrum above with the aliased power near $\sim$$60$ day$^{-1}$, we get something indistinguishable from the input signal:

In [ ]:
new_ift = dimensionless_ift(new_dft)
plt.figure(figsize = (3, 3))
plt.plot(times, new_signal)
plt.plot(times, new_ift)
plt.xlabel(r"Time [days]")
plt.ylabel(r"Reconstructed Signal $f(t)$")
plt.show()

An obvious symptom of aliasing is significant power near the Nyquist limit $k_c$ of the frequency spectrum. Functions with oscillation modes that strictly lie in the $[-k_c, +k_c]$ interval are said to be *bandwidth limited.*

If we filter out this frequency mode, we get the original "not-so-simple-sinusoid" as expected:

In [ ]:
truncated_dft = dft[:]
midpt = len(dft) // 2
for i in range(70, midpt):
    truncated_dft[midpt + i] = 0
    truncated_dft[midpt - i] = 0
smoothed = [_.real for _ in dimensionless_ift(truncated_dft)]

In [ ]:
fig, [ax1, ax2] = plt.subplots(figsize = (6, 3), nrows = 1, ncols = 2)
ax1.set_xlabel(r"$k$ [day$^{-1}$]")
ax1.set_ylabel(r"$\left|\tilde f(k)\right|$")
ax1.set_xlim([-80, 80])
ax2.set_xlabel(r"Time [days]")
ax2.set_ylabel(r"Signal")
ax1.plot(frequencies, [abs(_) for _ in new_dft], c = 'k')
ax2.plot(times, new_signal, c = 'k')
ax1.plot(frequencies, [abs(_) for _ in truncated_dft], c = 'r')
ax2.plot(times, smoothed, c = 'r')
plt.tight_layout()
plt.show()

However, if we hold onto the belief that this high-frequency oscillation is some crazy thing actually happening to a variable star, then our only hope is to sample the flux at even higher cadence such that the actual oscillation frequency is within the Nyquist limit.

## Windowing

You may have noticed that the frequency spectra we have seen are finite-width peaks as opposed to true delta functions. To bring out the non-zero power at these intermediate frequencies, let's examine the Fourier transform of our input sinusoid on a log-scale:

In [ ]:
plt.figure(figsize = (3, 3))
plt.plot(frequencies, absdft, c = 'k')
plt.xlabel(r"$k$ [day$^{-1}$]")
plt.ylabel(r"$\left|\tilde f(k)\right|$")
plt.xlim([-20, 20])
plt.yscale("log")
plt.show()

Leakage arises for a combination of reasons.

1. The frequencies of the input waves (1 day$^{-1}$, 3.6 day$^{-1}$, and 14.9 day$^{-1}$) do not *all* line up with an integer multiple of our sampling interval $\Delta t = 3 / 1500 = 0.002$ days $\approx 3$ minutes.

2. We have not measured an infinitely repeating periodic signal; we have measured a finite subset of a supposedly infinite function.

3. We have technically not measured a smooth function of time. We have *sampled* a continuous function.

The underlying functional form of our "data" is much better described as:

$$
f(t) = f_\infty(t) g(t),
$$

where

$$
f_\infty(t) = \sum_{i = 1}^3 A_i \sin (2\pi t / T_i)
$$

represents the three-component sinusoid that we have defined as our signal, with amplitudes $A_i$ and periods $T_i$, and

$$
g(t) = \begin{cases}
1 & (0 \leq t \leq 3)
\\
0 & (t < 0~\text{or}~t > 3)
\end{cases}
$$

describes the time-interval that we were collecting data.

For many purposes, it may be desirable to remove or "dampen" the power in the frequency domain that leaks between discrete modes. "Windowing" is the technique that achieves this goal. The basic idea behind windowing is that you multiply the input data $\lbrace f(q\Delta t)\rbrace = \lbrace f(0), f(\Delta t), f(2\Delta t), ..., f((N - 1)\Delta t)\rbrace$ by a "window function" that rises gradually from $q = 0$ to a maximum and then back to zero at $q = N - 1$.

Our dimensionless discrete Fourier transform from earlier then becomes

$$
\tilde f_n \rightarrow \sum_{q = 0}^{N - 1} f_q w_q e^{2\pi i q n / N}
$$

An implementation of this discrete Fourier transform:

In [ ]:
def windowed_dimensionless_dft(arr, windowfunc = None):
    r"""
    Compute a dimensionless discrete Fourier transform of the input
    array with the option for windowing.

    Parameters
    ----------
    arr : array-like
        The sequence of numbers to compute the discrete Fourier
        transform of.
    windowfunc : `callable`
        The function to use as the window in the Fourier transform.

    Returns
    -------
    An array of the same length as `arr` representing its Fourier
    transform, windowed according to the specified function.
    """
    ddft = []
    for idx in range(len(arr)):
        n = -len(arr) / 2 + idx
        ddft.append(0)
        for q in range(len(arr)):
            w = windowfunc(q) if windowfunc is not None else 1
            ddft[-1] += w * arr[q] * np.exp(-2j * np.pi * q * n /
                len(arr))
    return ddft

There are a number of common window functions. At this point, there are sufficiently many named after someone that trying to remember all of them is futile. Here are some of the more common ones:

$$
\begin{align}
w(q) &= \begin{cases}
1 & (\left|q\right| \leq N) \qquad \text{"Square window"}
\\
0 & (\left|q\right| > N)
\end{cases}
\\
w(q) &= 1 - \left|
\frac{q - N / 2}{ (N - 1) / 2}
\right| \qquad \text{"Bartlett Window"}
\\
w(q) &= \frac{1}{2} \left(
1 - \cos \left(2 \pi q / N
\right) \right) \qquad \text{"Hann Window"}
\\
\\
w(q) &= 1 - \left( \frac{2q - N}{N} \right)^2 \qquad \text{"Welch Window"}
\end{align}
$$

An implementation of each of these window functions:

In [ ]:
def square_window(q):
    r"""
    A "square" window, which is essentially a top-hat function of
    the bin offset `q`.
    """
    return int(abs(q) <= 500)

def bartlett_window(q):
    r"""
    The Barlett window as a function of the bin offset `q`.
    """
    return 1 - np.abs((q - len(signal) // 2) / ((len(signal) - 1) // 2))

def hann_window(q):
    r"""
    The Hann window as a function of the bin offset `q`.
    """
    return 0.5 * (1 - np.cos(2 * np.pi * q / len(signal)))

def welch_window(q):
    r"""
    The Welch window as a function of the bin offset `q`.
    """
    return 1 - ((2 * q - len(signal)) / len(signal))**2

A quick plot of each of these windows over our observation range:

In [ ]:
names = ["Square", "Bartlett", "Hann", "Welch"]
colors = ['r', 'b', 'r', 'b']
linestyles = ['-', '-', '--', '--']

In [ ]:
plt.figure(figsize = (3, 3))
for i, window in enumerate([square_window, bartlett_window,
    hann_window, welch_window]):
    plt.plot(range(len(signal)),
        [window(q) for q in range(len(signal))],
        c = colors[i], linestyle = linestyles[i],
        label = names[i])
plt.xlabel(r"Bin number")
plt.ylabel(r"Window function")
plt.legend()

Applying these windows to our Fourier transform leads to a visible reduction in the amount of leakage:

In [ ]:
plt.figure(figsize = (3, 3))
plt.plot(frequencies, absdft, c = 'k', label = "Original")
for i, window in enumerate([square_window, bartlett_window,
    hann_window, welch_window]):
    plt.plot(frequencies,
    [abs(_) for _ in windowed_dimensionless_dft(signal,
        windowfunc = window)],
    c = colors[i], linestyle = linestyles[i], label = names[i])
plt.xlim([-20, 20])
plt.yscale("log")
plt.legend(loc = "lower right", ncol = 2)
plt.xlabel(r"$k$ [day$^{-1}$]")
plt.ylabel(r"$\left|\tilde{f}(k)\right|$")
plt.show()

The Hann window removed the most leakage in this example. There is no window that is perfect for every application; this is also a big reason why there are so many windows. Hann is generally a good place to start though, for a key reason.

The Square and Bartlett windows have introduced some amount of "waviness" into this frequency spectrum. Can you guess why? *Hint*: these windows have a feature that the Hann window can be thought of as the "opposite" of.

## Fourier Transforms in Two Dimensions

The Fourier transform extends to two-dimensions in a straight-forward way. For a function $f(x_1, x_2)$, where $k_1$ and $k_2$ denote the component frequencies of $x_1$ and $x_2$, the Fourier transform and its inverse can be written as:

$$
\begin{align}
\tilde f(k_1, k_2) &= \int_{-\infty}^\infty \int_{-\infty}^\infty f(x_1, x_2) e^{-2\pi i (k_1 x_1 + k_2 x_2)} dx_1 dx_2
\\
f(x_1, x_2) &= \int_{-\infty}^\infty \int_{-\infty}^\infty \tilde f(k_1, k_2) e^{2\pi i (k_1 x_1 + k_2 x_2)} dk_1 dk_2.
\end{align}
$$

If the $f(x_1, x_2)$ is a separable function, meaning that it can be re-written as $f(x_1, x_2) = g(x_1)h(x_2)$, then these integrals are also separable:

$$
\begin{align}
\tilde f(k_1, k_2) &= \int_{-\infty}^\infty g(x_1) e^{-2\pi i k_1 x_1} dx_1 \int_{-\infty}^\infty h(x_2) e^{-2\pi i k_2 x_2} dx_2
\\
&= \tilde g(x_1) \tilde h(x_2)
\\
f(x_1, x_2) &= \int_{-\infty}^\infty \tilde g(k_1) e^{2\pi i k_1 x_1} dk_1 \int_{-\infty}^\infty \tilde h(x_2) e^{2\pi i k_2 x_2} dk_2
\\
&= g(x_1) h(x_2)
\end{align}
$$

Numerically, the discrete Fourier transform in 2-D follows from a similar derivation as the 1-D transform:

$$
\begin{align}
\tilde{f}(k_1, k_2) &\approx \sum_{q_2 = 0}^{N_2 - 1} \sum_{q_1 = 0}^{N_1 - 1} f(x_1, x_2) e^{2\pi i (k_1 x_1 + k_2 x_2)} \Delta x_1 \Delta x_2
\\
&\rightarrow \sum_{q_2 = 0}^{N_2 - 1} \sum_{q_1 = 0}^{N_1 - 1} f(x_1, x_2) e^{2\pi i (q_1 n_1 / N_1 + q_2 n_2 / N_2)}
\end{align}
$$

And the inverse transforms:

$$
\begin{align}
f(x_1, x_2) &\approx \sum_{q_2 = 0}^{N_2 - 1} \sum_{q_1 = 0}^{N_1 - 1} \tilde{f}(k_1, k_2) e^{2\pi i (k_1 x_1 + k_2 x_2)} \Delta k_1 \Delta k_2
\\
&\rightarrow \sum_{q_2 = 0}^{N_2 - 1} \sum_{q_1 = 0}^{N_1 - 1} \tilde{f}(k_1, k_2) e^{2\pi i (q_1 n_1 / N_1 + q_2 n_2 / N_2)}
\end{align}
$$

Intuitively, a 2-D Fourier transform is essentially "a Fourier transform within a Fourier transform."

Our toy sinusoids are small enough that discrete Fourier transforms are still fast enough to do compute "by-hand." Evaluating this double summation directly, however, is surprisingly slow due to its harsh $\mathcal{O}(N^4)$ scaling.

It is mathematically equivalent and much faster to evaluate a 1-D Fourier transforms along both axes (i.e., along the rows and then along the columns).

In [ ]:
def dimensionless_dft_2d(arr):
    r"""
    2-D dimensionless Fourier transform.

    Parameters
    ----------
    arr : array-like
        
    """
    # step 1: 1-D DFT to each row.
    ddft2d = [dimensionless_dft(row) for row in arr]

    # step 2: 1-D DFT to each column.
    ddft2d = [list(_) for _ in zip(*ddft2d)] # transpose
    ddft2d = [dimensionless_dft(col) for col in ddft2d]
    ddft2d = [list(_) for _ in zip(*ddft2d)] # transpose

    return ddft2d

Let's create a fake 2-D periodic signal! I'll create a second multi-component sinusoid and multiply the two.

In [ ]:
def second_odd_sinusoid(t):
    r"""
    Another three-component sinusoid.
    """
    return 0.8 * np.sin(4.7 * np.pi * t) + 0.5 * np.sin(
        1.3 * np.pi * t) + 0.15 * np.cos(8.9 * np.pi * t)

In [ ]:
plt.figure(figsize = (3, 3))
x1 = np.linspace(0, 5, 200)
x2 = np.linspace(0, 5, 200)
y = np.zeros((len(x1), len(x2)))
for i1 in range(len(x1)):
    for i2 in range(len(x2)):
        y[i1][i2] = not_so_simple_sinusoid(
            x1[i1]) * second_odd_sinusoid(x2[i2])
# rotate 90 degrees for imshow -- aligns "sides" with proper axis
pixels = [list(row) for row in zip(*y)][::-1]
plt.imshow(pixels, extent = (x1[0], x1[-1], x2[0], x2[-1]), cmap = "bwr")

It's Fourier transform and absolute value:

In [ ]:
ddft2d = dimensionless_dft_2d(y)
for i in range(len(ddft2d)):
    for j in range(len(ddft2d[i])):
        ddft2d[i][j] = abs(ddft2d[i][j])

To plot its Fourier transform, we'll use a colormap to show it across both dimensions in frequency:

In [ ]:
dx1 = x1[1] - x1[0]
dx2 = x2[1] - x2[0]
k1 = [(-len(y) / 2 + _) / (len(y) * dx1) for _ in range(len(y))]
k2 = [(-len(y[0]) / 2 + _) / (len(y[0]) * dx2) for _ in range(len(y[0]))]
pixels = [list(row) for row in zip(*ddft2d)][::-1] # rotate 90 degrees for imshow -- aligns "sides" with proper axis
plt.figure(figsize = (3, 3))
plt.imshow([np.log10(d) for d in pixels], extent = (k1[0], k1[-1], k2[0], k2[-1]), cmap = "Greys")
plt.xlabel(r"$k_1$ [day$^{-1}$]")
plt.ylabel(r"$k_2$ [day$^{-1}$]")
plt.colorbar(label = r"$\log_{10}\left|\tilde{f}(k_1, k_2)\right|$")
plt.show()

Here's another example with a cross-terms (i.e., $x_1$ times $x_2$):

In [ ]:
x1 = np.linspace(0, 5, 200)
x2 = np.linspace(0, 5, 200)
new_y = np.zeros((len(x1), len(x2)))
for i1 in range(len(x1)):
    for i2 in range(len(x2)):
        new_y[i1][i2] = not_so_simple_sinusoid(x1[i1]) * np.sin(x1[i1] * x2[i2])
pixels = [list(row) for row in zip(*new_y)][::-1] # rotate 90 degrees for imshow -- aligns "sides" with proper axis
plt.imshow(pixels, extent = (x1[0], x1[-1], x2[0], x2[-1]), cmap = "bwr")

And its Fourier transform:

In [ ]:
new_ddft2d = dimensionless_dft_2d(new_y)
for i in range(len(new_ddft2d)):
    for j in range(len(new_ddft2d[i])):
        new_ddft2d[i][j] = abs(new_ddft2d[i][j])

In [ ]:
dx1 = x1[1] - x1[0]
dx2 = x2[1] - x2[0]
k1 = [(-len(new_y) / 2 + _) / (len(new_y) * dx1) for _ in range(len(new_y))]
k2 = [(-len(new_y[0]) / 2 + _) / (len(new_y[0]) * dx2) for _ in range(len(new_y[0]))]
pixels = [list(row) for row in zip(*new_ddft2d)][::-1] # rotate 90 degrees for imshow -- aligns "sides" with proper axis
plt.figure(figsize = (3, 3))
plt.imshow([np.log10(d) for d in pixels], extent = (k1[0], k1[-1], k2[0], k2[-1]), cmap = "Greys")
plt.xlabel(r"$k_1$ [day$^{-1}$]")
plt.ylabel(r"$k_2$ [day$^{-1}$]")
plt.colorbar(label = r"$\log_{10}\left|\tilde{f}(k_1, k_2)\right|$")
plt.show()